# Enterprise B2B Company Search - V6 End-to-End Demo
Welcome to the `scaling-succotash` platform showcase. This notebook covers data ingestion, baseline deterministic search, intelligent semantic search, agentic routing, and the Singleflight Request Collapsing / Redis Cache optimizations.

## 1. Environment Overview & Imports
Let's import our tools and point them at the backend.

In [1]:
import time

import httpx
import polars as pl

API_URL = "http://127.0.0.1:8000/api/v2/search"
client = httpx.AsyncClient(timeout=30.0)

## 2. Data Exploration
We can inspect the raw data being ingested into OpenSearch using `polars`.

In [2]:
df = pl.read_csv("../data/companies.csv")
print(f"Total companies: {len(df)}")
df.head(5).to_pandas()

Total companies: 4


,,name,domain,year founded,industry,size range,locality,country,linkedin url,current employee estimate,total employee estimate
0,5872184,ibm,ibm.com,1911.0,information technology and services,10001+,"new york, new york, united states",united states,linkedin.com/company/ibm,274047,716906
1,4425416,tata consultancy services,tcs.com,1968.0,information technology and services,10001+,"bombay, maharashtra, india",india,linkedin.com/company/tata-consultancy-services,190771,341369
2,21074,accenture,accenture.com,1989.0,information technology and services,10001+,"dublin, dublin, ireland",ireland,linkedin.com/company/accenture,190689,455768
3,101,test company,test.com,NaN,software,1-10,"san francisco, california",united states,None,10,10


## 3. Deterministic Search (Baseline)
Traditional keyword filtering using strict matching.

In [3]:
payload = {"industry": "Software", "size": 10, "page": 1}
response = await client.post(API_URL, json=payload)
print(response.json())

{'results': [{'name': 'test company', 'domain': 'test.com', 'industry': 'software', 'locality': 'san francisco, california', 'country': 'united states', 'size_range': '1-10', 'year_founded': 0, 'tags': [], 'embedding': [0.027810823172330856, -0.016888698562979698, -0.01434830017387867, -0.04003024101257324, -0.010510686784982681, -0.08697016537189484, -0.07193343341350555, 0.013724146410822868, -0.011689256876707077, -0.0061881402507424355, 0.035844992846250534, -0.02668090909719467, -0.010950478725135326, -0.027868671342730522, 0.013432544656097889, -0.028109604492783546, -0.008846845477819443, -0.08803045004606247, 0.12946830689907074, -0.059006061404943466, -0.0007113370229490101, -0.07176000624895096, -0.05952756851911545, -0.008407744579017162, 0.05958881601691246, -0.01543587725609541, 0.0006419218261726201, 0.06683434545993805, 0.014208250679075718, -0.05488596856594086, -0.02189018949866295, 0.015223282389342785, 0.11372143775224686, 0.027389228343963623, 0.11079664528369904, -

In [8]:
from pprint import pprint

pprint(response.json())

{'results': [{'country': 'united states',
              'domain': 'test.com',
              'embedding': [0.027810823172330856,
                            -0.016888698562979698,
                            -0.01434830017387867,
                            -0.04003024101257324,
                            -0.010510686784982681,
                            -0.08697016537189484,
                            -0.07193343341350555,
                            0.013724146410822868,
                            -0.011689256876707077,
                            -0.0061881402507424355,
                            0.035844992846250534,
                            -0.02668090909719467,
                            -0.010950478725135326,
                            -0.027868671342730522,
                            0.013432544656097889,
                            -0.028109604492783546,
                            -0.008846845477819443,
                            -0.08803045004606247,
             

In [13]:
print(type(response.json()))

<class 'dict'>


In [15]:
pprint([(r["country"], r["domain"]) for r in response.json()["results"]])

[('united states', 'test.com')]


## 4. Intelligent Semantic Search
Fuzzy query intent extraction and vector hnsw embeddings.

In [4]:
payload = {"query": "cloud providers supporting kubernetes"}
response = await client.post(f"{API_URL}/intelligent", json=payload)
print(response.json())

{'results': [{'name': 'test company', 'domain': 'test.com', 'industry': 'software', 'locality': 'san francisco, california', 'country': 'united states', 'size_range': '1-10', 'year_founded': 0, 'tags': [], 'embedding': [0.027810823172330856, -0.016888698562979698, -0.01434830017387867, -0.04003024101257324, -0.010510686784982681, -0.08697016537189484, -0.07193343341350555, 0.013724146410822868, -0.011689256876707077, -0.0061881402507424355, 0.035844992846250534, -0.02668090909719467, -0.010950478725135326, -0.027868671342730522, 0.013432544656097889, -0.028109604492783546, -0.008846845477819443, -0.08803045004606247, 0.12946830689907074, -0.059006061404943466, -0.0007113370229490101, -0.07176000624895096, -0.05952756851911545, -0.008407744579017162, 0.05958881601691246, -0.01543587725609541, 0.0006419218261726201, 0.06683434545993805, 0.014208250679075718, -0.05488596856594086, -0.02189018949866295, 0.015223282389342785, 0.11372143775224686, 0.027389228343963623, 0.11079664528369904, -

In [17]:
pprint([(r["country"], r["domain"]) for r in response.json()["results"]])

[('united states', 'test.com')]


## 5. Agentic Workflow
Queries with external context automatically get routed to `requires_agent: true`.

In [18]:
payload = {"query": "latest acquisitions by microsoft in ai"}
response = await client.post(f"{API_URL}/intelligent", json=payload)
print(response.json())

{'results': [{'name': 'test company', 'domain': 'test.com', 'industry': 'software', 'locality': 'san francisco, california', 'country': 'united states', 'size_range': '1-10', 'year_founded': 0, 'tags': [], 'embedding': [0.027810823172330856, -0.016888698562979698, -0.01434830017387867, -0.04003024101257324, -0.010510686784982681, -0.08697016537189484, -0.07193343341350555, 0.013724146410822868, -0.011689256876707077, -0.0061881402507424355, 0.035844992846250534, -0.02668090909719467, -0.010950478725135326, -0.027868671342730522, 0.013432544656097889, -0.028109604492783546, -0.008846845477819443, -0.08803045004606247, 0.12946830689907074, -0.059006061404943466, -0.0007113370229490101, -0.07176000624895096, -0.05952756851911545, -0.008407744579017162, 0.05958881601691246, -0.01543587725609541, 0.0006419218261726201, 0.06683434545993805, 0.014208250679075718, -0.05488596856594086, -0.02189018949866295, 0.015223282389342785, 0.11372143775224686, 0.027389228343963623, 0.11079664528369904, -

In [19]:
pprint([(r["country"], r["domain"]) for r in response.json()["results"]])

[('united states', 'test.com')]


## 6. Performance Optimization & Cache Stampede Prevention
Testing the V5 optimizations sequentially. Notice the `X-Cache-Hit: true` response header!

In [20]:
query = "healthcare startups in london"
payload = {"query": query}

start = time.time()
r1 = await client.post(f"{API_URL}/intelligent", json=payload)
t1 = time.time() - start
print(f"First Request (Full Pipeline): {t1:.4f}s - Cache Hit: {r1.headers.get('x-cache-hit', 'false')}")

start = time.time()
r2 = await client.post(f"{API_URL}/intelligent", json=payload)
t2 = time.time() - start
print(f"Second Request (Redis Cache): {t2:.4f}s - Cache Hit: {r2.headers.get('x-cache-hit', 'false')}")

First Request (Full Pipeline): 0.5737s - Cache Hit: true
Second Request (Redis Cache): 0.0055s - Cache Hit: true


In [ ]:
await client.aclose()